In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from utils import (download_sse_stock_data, download_szse_stock_data,
                   read_sse_stock_data, read_szse_stock_data,
                   download_stock_industry_data, read_stock_industry_data,
                   get_current_date_str)


def filter_stocks(df, rise_or_fall, pct_chg_thresh, amount_thresh, ascending):

    if rise_or_fall == 'rise':
        filtered = df[(df['pctChg'] >= pct_chg_thresh) & (df['amount'] >= amount_thresh)]
    else:
        filtered = df[(df['pctChg'] <= -pct_chg_thresh) & (df['amount'] >= amount_thresh)]

    filtered_df = filtered.sort_values('pctChg', ascending=ascending)
    return filtered_df[['date', 'code', 'name', 'pctChg', 'amount']]

date_str = get_current_date_str()
# date_str = '2026-02-27'

download_stock_industry_data(date_str)
download_sse_stock_data(date_str)
download_szse_stock_data(date_str)

stock_industry_df = read_stock_industry_data(date_str)
sse_df = read_sse_stock_data(date_str)
szse_df = read_szse_stock_data(date_str)

sse_df = sse_df.merge(stock_industry_df, on='code', how='left')
szse_df = szse_df.merge(stock_industry_df, on='code', how='left')

sse_df_rise = filter_stocks(sse_df, 'rise', 8.00, 8e8, ascending=False)
sse_df_fall = filter_stocks(sse_df, 'fall', 8.00, 8e8, ascending=True)

szse_df_rise = filter_stocks(szse_df, 'rise', 8.00, 8e8, ascending=False)
szse_df_fall = filter_stocks(szse_df, 'fall', 8.00, 8e8, ascending=True)

In [ ]:
def format_stock_list(df):
    result = []
    for _, row in df.iterrows():
        name = row['name']
        pct_chg = row['pctChg']
        amount = row['amount'] / 1e8  # 转换为亿元
        item = f"{name}（{pct_chg:+.2f}%，{amount:.2f}亿）"
        result.append(item)
    return result


def make_inner_table(stock_list, cols=5):
    """将股票列表转为 cols 列的内嵌 HTML 表格"""
    if not stock_list:
        return "<span style='color:gray'>无</span>"
    
    rows = [stock_list[i:i+cols] for i in range(0, len(stock_list), cols)]
    html = "<table style='border-collapse:collapse; font-size:12px;'>"
    for row in rows:
        html += "<tr>"
        for cell in row:
            html += f"<td style='padding:3px 6px; white-space:nowrap;'>{cell}</td>"
        # 补齐不足 cols 的最后一行
        for _ in range(cols - len(row)):
            html += "<td></td>"
        html += "</tr>"
    html += "</table>"
    return html


def make_outer_table(date_str, sse_rise_list, sse_fall_list, szse_rise_list, szse_fall_list):
    th_style = "border:1px solid #ccc; padding:6px 10px; background:#f5f5f5; text-align:center;"
    td_style = "border:1px solid #ccc; padding:6px 10px; vertical-align:top; text-align:center;"
    td_inner = "border:1px solid #ccc; padding:4px; vertical-align:top;"

    html = f"""
    <table style='border-collapse:collapse; font-size:13px;'>
        <thead>
            <tr>
                <th style='{th_style}'>{date_str}</th>
                <th style='{th_style}'>涨幅超8%成交超8亿</th>
                <th style='{th_style}'>跌幅超8%成交超8亿</th>
            </tr>
        </thead>
        <tbody>
            <tr>
                <td style='{td_style}'>上证</td>
                <td style='{td_inner}'>{make_inner_table(sse_rise_list)}</td>
                <td style='{td_inner}'>{make_inner_table(sse_fall_list)}</td>
            </tr>
            <tr>
                <td style='{td_style}'>深证</td>
                <td style='{td_inner}'>{make_inner_table(szse_rise_list)}</td>
                <td style='{td_inner}'>{make_inner_table(szse_fall_list)}</td>
            </tr>
        </tbody>
    </table>
    """
    return html

sse_rise_list  = format_stock_list(sse_df_rise)
sse_fall_list  = format_stock_list(sse_df_fall)
szse_rise_list = format_stock_list(szse_df_rise)
szse_fall_list = format_stock_list(szse_df_fall)
html_output = make_outer_table(date_str, sse_rise_list, sse_fall_list, szse_rise_list, szse_fall_list)
display(HTML(html_output))

In [ ]:
all_codes = pd.concat([sse_df_rise, sse_df_fall, szse_df_rise, szse_df_fall])['code'].tolist()
print(','.join(all_codes))

In [ ]:

# 设置matplotlib支持中文显示
plt.rcParams['font.sans-serif'] = ['Songti SC', 'Heiti TC', 'DejaVu Sans']

# 1. 合并数据并计算总成交额
all_df = pd.concat([sse_df, szse_df], ignore_index=True)
total_amount = all_df['amount'].sum()

# 2. 排序并筛选
all_df_sorted = all_df.sort_values('pctChg', ascending=False)
above_5pct_df = all_df_sorted[all_df_sorted['pctChg'] > 5.0]
top_10_percent_df = all_df_sorted.head(int(len(all_df_sorted) * 0.1))


def plot_industry_summary(summary_df, title):
    # 按score降序排列
    summary_df = summary_df.sort_values('score', ascending=False)
    industries = summary_df['industry']
    x = np.arange(len(industries))

    # 归一化
    stock_count_norm = summary_df['stock_count'] / 100
    avg_pctChg_norm = summary_df['avg_pctChg'] / 10
    industry_amount_ratio_norm = summary_df['industry_amount_ratio'] / 1

    width = 0.25

    fig, ax = plt.subplots(figsize=(14, 6))
    rects1 = ax.bar(x - width, stock_count_norm, width, label='stock_count/100')
    rects2 = ax.bar(x, avg_pctChg_norm, width, label='avg_pctChg/10')
    rects3 = ax.bar(x + width, industry_amount_ratio_norm, width, label='industry_amount_ratio/1')

    # 在每个柱子上添加数值标签
    for i, rect in enumerate(rects1):
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2., height + 0.01,
                f'{int(summary_df["stock_count"].iloc[i])}', ha='center', va='bottom', fontsize=8)

    for i, rect in enumerate(rects2):
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2., height + 0.01,
                f'{summary_df["avg_pctChg"].iloc[i]:.2f}', ha='center', va='bottom', fontsize=8)

    for i, rect in enumerate(rects3):
        height = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2., height + 0.01,
                f'{summary_df["industry_amount_ratio"].iloc[i]:.2%}', ha='center', va='bottom', fontsize=8)

    ax.set_ylabel('Normalized Value')
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(industries, rotation=30, ha='right', fontsize=10)
    ax.legend()
    plt.tight_layout()
    plt.show()

# 3. 按industry汇总
def industry_summary(df, total_amount):

    grouped = df.groupby('industry').agg(
        stock_count=('code', 'count'),
        avg_pctChg=('pctChg', 'mean'),
        _industry_amount=('amount', 'sum')
    ).reset_index()

    grouped['industry_amount_ratio'] = grouped['_industry_amount'] / total_amount
    grouped = grouped.drop(columns=['_industry_amount'])

    grouped['score'] = grouped['stock_count'] * grouped['avg_pctChg'] * grouped['industry_amount_ratio']
    stock_list_series = df.groupby('industry').apply(lambda g: "<br>".join([f"<span style='color:blue; font-family: monospace'>{row['name'].ljust(g['name'].str.len().max())}</span> ({row['pctChg']:.2f}% {row['amount']/1e8:.2f}亿元)".replace(' ', '&nbsp;') for _, row in g.iterrows()]))
    grouped['stock_list'] = grouped['industry'].map(stock_list_series)
    grouped = grouped.sort_values('score', ascending=False).head(10)

    return grouped

above_5pct_summary = industry_summary(above_5pct_df, total_amount)
top_10_percent_summary = industry_summary(top_10_percent_df, total_amount)


plot_industry_summary(above_5pct_summary, "涨幅超过5%行业分布")
plot_industry_summary(top_10_percent_summary, "Top 10%涨幅行业分布")